# 03 Relation Extraction and Graph Construction

This notebook is self-contained. It builds the core symbol-emotion relation by connecting each symbol to the nearest emotion word in the same poem when the token distance is at most 10.

This is the central research method of the project. The graph does not classify whole poems. Instead, it looks for local poetic associations: a symbol is linked to an emotion only when an emotion word appears very nearby in the poem text.


In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
GRAPHS_DIR = PROJECT_ROOT / 'outputs' / 'graphs'
POEMS_CLEAN_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
RELATIONS_PATH = PROCESSED_DIR / 'symbol_emotion_edges.csv'
GRAPH_NODES_PATH = PROCESSED_DIR / 'graph_nodes.csv'
GRAPH_EDGES_PATH = PROCESSED_DIR / 'graph_edges.csv'
GRAPH_JSON_PATH = GRAPHS_DIR / 'poetry_graph.json'
MAX_DISTANCE = 10
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

## Functions

The relation means symbol appears near emotion-related language in this poem, not symbol always means this emotion.

The short distance threshold of 10 tokens is intentionally conservative. A larger window may create more edges, but it also risks connecting words that are not actually related in the poem. The notebook also stores evidence snippets so every edge can be inspected.


In [2]:
def token_context(text, start_token, end_token, window=10):
    tokens = text.split()
    start = max(0, min(start_token, end_token) - window)
    end = min(len(tokens), max(start_token, end_token) + window + 1)
    return ' '.join(tokens[start:end])

def closest_emotions_for_symbol(symbol_row, emotion_rows, max_distance=MAX_DISTANCE):
    if emotion_rows.empty:
        return []
    candidates = emotion_rows.copy()
    candidates['token_distance'] = (candidates['start_token'] - symbol_row.start_token).abs()
    candidates['line_distance'] = (candidates['line_number'] - symbol_row.line_number).abs()
    candidates = candidates[candidates['token_distance'] <= max_distance]
    if candidates.empty:
        return []
    candidates = candidates[candidates['token_distance'] == candidates['token_distance'].min()]
    candidates = candidates[candidates['line_distance'] == candidates['line_distance'].min()]
    return candidates.to_dict('records')

def extract_symbol_emotion_edges(symbols_df, emotions_df, poems_df, max_distance=MAX_DISTANCE):
    poem_text = poems_df.set_index('poem_id')['poem_text'].to_dict()
    rows = []
    for poem_id, symbol_group in symbols_df.groupby('poem_id'):
        emotion_group = emotions_df[emotions_df['poem_id'] == poem_id]
        for symbol in symbol_group.itertuples(index=False):
            for emotion in closest_emotions_for_symbol(symbol, emotion_group, max_distance=max_distance):
                distance = abs(int(symbol.start_token) - int(emotion['start_token']))
                line_distance = abs(int(symbol.line_number) - int(emotion['line_number']))
                rows.append({'source_symbol': symbol.symbol, 'target_emotion': emotion['emotion_category'], 'poem_id': poem_id, 'title': symbol.title, 'author': symbol.author, 'symbol_token': int(symbol.start_token), 'emotion_token': int(emotion['start_token']), 'token_distance': distance, 'line_distance': line_distance, 'symbol_line': int(symbol.line_number), 'emotion_line': int(emotion['line_number']), 'context_snippet': token_context(poem_text.get(poem_id, ''), int(symbol.start_token), int(emotion['start_token']))})
    columns = ['source_symbol', 'target_emotion', 'poem_id', 'title', 'author', 'symbol_token', 'emotion_token', 'token_distance', 'line_distance', 'symbol_line', 'emotion_line', 'context_snippet']
    return pd.DataFrame(rows, columns=columns)

def limited_values(series, limit=5):
    values = []
    for value in series.dropna().astype(str):
        if value and value not in values:
            values.append(value)
        if len(values) == limit:
            break
    return ' | '.join(values)

def aggregate_symbol_emotion_edges(relations_df):
    grouped = relations_df.groupby(['source_symbol', 'target_emotion'], as_index=False).agg(frequency=('poem_id', 'size'), poem_count=('poem_id', 'nunique'), avg_distance=('token_distance', 'mean'), min_distance=('token_distance', 'min'), example_titles=('title', limited_values), example_snippets=('context_snippet', limited_values))
    grouped['score'] = grouped['frequency'] / grouped['avg_distance'].clip(lower=1)
    return grouped.sort_values(['frequency', 'poem_count'], ascending=False)

def build_graph_nodes_edges(poems_df, symbols_df, emotions_df, relations_df):
    node_rows = []
    edge_rows = []
    for symbol, frequency in symbols_df['symbol'].value_counts().items():
        node_rows.append({'id': f'symbol:{symbol}', 'label': symbol, 'type': 'symbol', 'frequency': int(frequency), 'author': ''})
    for emotion, frequency in emotions_df['emotion_category'].value_counts().items():
        node_rows.append({'id': f'emotion:{emotion}', 'label': emotion, 'type': 'emotion', 'frequency': int(frequency), 'author': ''})
    for poem in poems_df.itertuples(index=False):
        node_rows.append({'id': f'poem:{poem.poem_id}', 'label': poem.title or poem.poem_id, 'type': 'poem', 'frequency': 1, 'author': poem.author})
    for author, count in poems_df['author'].fillna('').value_counts().items():
        if author:
            node_rows.append({'id': f'author:{author}', 'label': author, 'type': 'author', 'frequency': int(count), 'author': ''})
    for row in aggregate_symbol_emotion_edges(relations_df).itertuples(index=False):
        edge_rows.append({'source': f'symbol:{row.source_symbol}', 'target': f'emotion:{row.target_emotion}', 'type': 'NEAR_EMOTION', 'weight': int(row.frequency), 'poem_id': '', 'evidence': row.example_snippets})
    for row in symbols_df.itertuples(index=False):
        edge_rows.append({'source': f'poem:{row.poem_id}', 'target': f'symbol:{row.symbol}', 'type': 'HAS_SYMBOL', 'weight': 1, 'poem_id': row.poem_id, 'evidence': ''})
    for row in emotions_df.itertuples(index=False):
        edge_rows.append({'source': f'poem:{row.poem_id}', 'target': f'emotion:{row.emotion_category}', 'type': 'HAS_EMOTION_WORD', 'weight': 1, 'poem_id': row.poem_id, 'evidence': row.matched_word})
    for poem in poems_df.itertuples(index=False):
        if poem.author:
            edge_rows.append({'source': f'poem:{poem.poem_id}', 'target': f'author:{poem.author}', 'type': 'WRITTEN_BY', 'weight': 1, 'poem_id': poem.poem_id, 'evidence': ''})
    return pd.DataFrame(node_rows).drop_duplicates('id'), pd.DataFrame(edge_rows).drop_duplicates(['source', 'target', 'type', 'poem_id'])

def export_graph_json(nodes_df, edges_df, path):
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump({'nodes': nodes_df.to_dict('records'), 'edges': edges_df.to_dict('records')}, handle, ensure_ascii=False, indent=2)

## Build Relations and Graph

The first output is symbol_emotion_edges.csv, which contains one row per local symbol-emotion relation with poem title, author, token distance, line distance, and context snippet.

The second output is the graph representation. It contains symbol nodes, emotion nodes, poem nodes, and author nodes. The most important edge type is NEAR_EMOTION, which aggregates repeated symbol-emotion relations across the corpus.


In [3]:
poems_df = pd.read_csv(POEMS_CLEAN_PATH)
symbols_df = pd.read_csv(SYMBOLS_PATH)
emotions_df = pd.read_csv(EMOTIONS_PATH)
relations_df = extract_symbol_emotion_edges(symbols_df, emotions_df, poems_df, max_distance=MAX_DISTANCE)
relations_df.to_csv(RELATIONS_PATH, index=False)
relations_df.head()

,source_symbol,target_emotion,poem_id,title,author,symbol_token,emotion_token,token_distance,line_distance,symbol_line,emotion_line,context_snippet
0,bone,mortality,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,1,1,0,0,1,1,"Dog bone, stapler, cribbage board, garlic pres..."
1,window,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,70,78,8,2,21,23,this window is split. It's dividing in two. Ve...
2,window,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,124,120,4,2,37,35,
3,dog,mortality,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,0,1,1,0,1,1,"Dog bone, stapler, cribbage board, garlic pres..."
4,baby,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,87,78,9,2,25,23,"Velvet moss, sagebrush, willow branch, robin's..."


In [4]:
nodes_df, edges_df = build_graph_nodes_edges(poems_df, symbols_df, emotions_df, relations_df)
nodes_df.to_csv(GRAPH_NODES_PATH, index=False)
edges_df.to_csv(GRAPH_EDGES_PATH, index=False)
export_graph_json(nodes_df, edges_df, GRAPH_JSON_PATH)
aggregate_symbol_emotion_edges(relations_df).head(20)

,source_symbol,target_emotion,frequency,poem_count,avg_distance,min_distance,example_titles,example_snippets,score
17721,light,hope,1864,887,0.000000,0,The One Thing That Can Save America | Stung | ...,counting? A mood soon to be forgotten In cross...,1864.000000
9289,dream,wonder,985,504,0.009137,0,"Invisible Fish | loose strife [Say, when we wo...",will come ashore and paint dreams on the dying...,985.000000
24701,rain,melancholy,734,407,0.000000,0,The New Church | scars | All | from Stone: 98 ...,"The apple trees have taken over the sky, seque...",734.000000
27288,shadow,fear,648,385,0.000000,0,West of Myself | (Demilitarized Zone) | The Ai...,"like a dog after dark, dragging a shadow you’v...",648.000000
15090,home,nostalgia,588,341,0.000000,0,hamsters are heads with little characteristics...,long: at home three waffle friends wait coolin...,588.000000
35565,wing,freedom,566,357,0.000000,0,Objects Used to Prop Open a Window | Umbrella ...,"skin and bones, black bat wing. We're alike, y...",566.000000
2921,bone,mortality,533,331,0.000000,0,Objects Used to Prop Open a Window | Umbrella ...,"Dog bone, stapler, cribbage board, garlic pres...",533.000000
29430,spirit,faith,516,255,0.000000,0,Don’t Bother the Earth Spirit | The Lamp of Mu...,Don’t bother the earth spirit who lives here. ...,516.000000
29431,spirit,transcendence,516,255,0.000000,0,Don’t Bother the Earth Spirit | The Lamp of Mu...,Don’t bother the earth spirit who lives here. ...,516.000000
509,angel,transcendence,436,223,0.000000,0,Japanese Poems | In an Artist's Studio | Sonne...,day. The angel in her credenza of extreme beau...,436.000000
